In [ ]:
import csv
import os
import sys
import pandas as pd
import numpy as np
from emmaemb.core import Emma
from emmaemb.vizualisation import get_knn_alignment_scores
from dim_reduction_utils import load_imbalanced_cryptic_and_regular_data
from constants import DATA_PATH, EMBEDDINGS_PATH, IMG_OUTPUT_PATH, EMB_SPACES, CRYPTOBENCH_TRAIN_DATASET, SCPDB_DATASET
sys.path.append('/home/unix/vkrhk/EmmaEmb/analysis')

METRIC = 'euclidean'
N_TREES = 50

emb_spaces = EMB_SPACES
datasets = [CRYPTOBENCH_TRAIN_DATASET, SCPDB_DATASET]


In [ ]:
for emb_space in emb_spaces:
    embeddings_name = emb_space[0]
    emma = load_imbalanced_cryptic_and_regular_data(emb_space, datasets, DATA_PATH)
    print(f"Finished calculating pairwise distances for {embeddings_name}.")
    emma.build_annoy_index(emb_space=embeddings_name, metric=METRIC, n_trees=N_TREES)
    df = get_knn_alignment_scores(emma, feature='binding_site', k=3, metric=METRIC, use_annoy=True, n_trees=N_TREES, annoy_metric=METRIC)
    heatmap_data = (
        df.groupby(["Class", "Embedding"])["Fraction"]
        .mean()
        .unstack()  # Reshape to have Classes as rows and Embeddings as columns
    )

    class_counts = df.groupby("Class").size()

    heatmap_data.index = [
        f"{feature_class} (n = {int(count / len(df['Embedding'].unique()))})"
        for feature_class, count in zip(
            heatmap_data.index, class_counts[heatmap_data.index]
        )
    ]

    heatmap_data.to_csv(f'{IMG_OUTPUT_PATH}/knn-binding-sites/{embeddings_name}.csv')


In [ ]:
results = {}
for filename in os.listdir(f'{IMG_OUTPUT_PATH}/knn-binding-sites'):
    if filename.endswith('.csv'):
        embedding_space = filename.split('_')[0]
        results[embedding_space] = {}
        df = pd.read_csv(f'{IMG_OUTPUT_PATH}/knn-binding-sites/{filename}', names=['BINDING-TYPE', 'KNN-ALIGNMENT-SCORE'], header=0)

        for i in df.index:
            binding_type = df.loc[i, 'BINDING-TYPE'].split(' ')[0]
            knn_alignment_score = df.loc[i, 'KNN-ALIGNMENT-SCORE']
            results[embedding_space][binding_type] = knn_alignment_score


results